# Flood Prediction System for Bihar and Flood-Prone Regions in India
## Machine Learning Model Development

**Project Overview:**
- Predict flood risk based on weather parameters
- Focus on Bihar and other flood-prone regions
- Input: Rainfall, Humidity, Temperature, River Level
- Output: Flood Risk (Low, Medium, High)

**Regions Covered:**
- Bihar (Patna, Muzaffarpur, Darbhanga, Sitamarhi)
- Assam (Guwahati, Dhubri)
- Uttar Pradesh (Gorakhpur, Ballia)
- West Bengal (Malda, Murshidabad)

## 1. Import Required Libraries

In [ ]:
# Data manipulation and analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Machine Learning libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve

# Model saving
import pickle
import joblib

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## 2. Data Collection - Creating Synthetic Dataset

Since historical flood data requires specialized sources, we'll create a realistic synthetic dataset based on:
- Actual flood patterns in Bihar and nearby regions
- Monsoon rainfall patterns (June-September)
- River discharge levels during flood seasons
- Historical flood occurrences in these regions

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Define regions and their characteristics
regions = {
    'Patna': {'flood_prone': 0.75, 'base_rainfall': 1200},
    'Muzaffarpur': {'flood_prone': 0.80, 'base_rainfall': 1300},
    'Darbhanga': {'flood_prone': 0.85, 'base_rainfall': 1400},
    'Sitamarhi': {'flood_prone': 0.82, 'base_rainfall': 1350},
    'Guwahati': {'flood_prone': 0.78, 'base_rainfall': 2800},
    'Dhubri': {'flood_prone': 0.83, 'base_rainfall': 2900},
    'Gorakhpur': {'flood_prone': 0.72, 'base_rainfall': 1100},
    'Ballia': {'flood_prone': 0.70, 'base_rainfall': 1050},
    'Malda': {'flood_prone': 0.68, 'base_rainfall': 1600},
    'Murshidabad': {'flood_prone': 0.71, 'base_rainfall': 1550}
}

# Generate dataset
n_samples = 5000
data = []

for _ in range(n_samples):
    region = np.random.choice(list(regions.keys()))
    region_data = regions[region]
    
    # Generate date (focus on monsoon season June-September for higher flood risk)
    year = np.random.randint(2015, 2025)
    month = np.random.choice([1,2,3,4,5,6,7,8,9,10,11,12], 
                            p=[0.03,0.03,0.05,0.05,0.08,0.15,0.20,0.20,0.15,0.04,0.02,0.00])
    day = np.random.randint(1, 29)
    date = f"{year}-{month:02d}-{day:02d}"
    
    # Monsoon indicator
    is_monsoon = 1 if month in [6, 7, 8, 9] else 0
    
    # Weather parameters
    # Rainfall (mm) - higher during monsoon
    if is_monsoon:
        daily_rainfall = np.random.gamma(shape=3, scale=40) + np.random.uniform(0, 100)
    else:
        daily_rainfall = np.random.gamma(shape=1.5, scale=8) + np.random.uniform(0, 20)
    
    # 7-day cumulative rainfall
    rainfall_7day = daily_rainfall * np.random.uniform(4, 7)
    
    # 15-day cumulative rainfall
    rainfall_15day = rainfall_7day * np.random.uniform(1.8, 2.5)
    
    # Humidity (%) - higher during monsoon
    if is_monsoon:
        humidity = np.random.normal(85, 8)
    else:
        humidity = np.random.normal(65, 12)
    humidity = np.clip(humidity, 30, 100)
    
    # Temperature (°C) - varies by season
    if month in [3, 4, 5]:  # Summer
        temperature = np.random.normal(35, 4)
    elif month in [12, 1, 2]:  # Winter
        temperature = np.random.normal(18, 4)
    else:  # Monsoon and post-monsoon
        temperature = np.random.normal(28, 3)
    temperature = np.clip(temperature, 10, 45)
    
    # River water level (meters above danger mark)
    # Influenced by rainfall
    base_level = -2 + (rainfall_7day / 100)
    river_level = base_level + np.random.normal(0, 0.5)
    
    # Soil moisture (%) - influenced by recent rainfall
    soil_moisture = min(100, 40 + (rainfall_7day / 10) + np.random.uniform(-10, 10))
    
    # Wind speed (km/h)
    wind_speed = np.random.gamma(shape=2, scale=8) + 5
    
    # Determine flood occurrence based on multiple factors
    flood_score = (
        (rainfall_7day > 250) * 30 +
        (rainfall_15day > 450) * 25 +
        (river_level > 0) * 20 +
        (humidity > 85) * 10 +
        (soil_moisture > 80) * 10 +
        is_monsoon * 5
    )
    
    # Add region-specific flood proneness
    flood_score *= region_data['flood_prone']
    
    # Add randomness
    flood_score += np.random.uniform(-15, 15)
    
    # Classify flood risk
    if flood_score > 60:
        flood_risk = 'High'
        flood_occurred = 1
    elif flood_score > 35:
        flood_risk = 'Medium'
        flood_occurred = np.random.choice([0, 1], p=[0.6, 0.4])
    else:
        flood_risk = 'Low'
        flood_occurred = 0
    
    # Append to data
    data.append({
        'Date': date,
        'Region': region,
        'Daily_Rainfall_mm': round(daily_rainfall, 2),
        'Rainfall_7Day_mm': round(rainfall_7day, 2),
        'Rainfall_15Day_mm': round(rainfall_15day, 2),
        'Humidity_%': round(humidity, 2),
        'Temperature_C': round(temperature, 2),
        'River_Level_m': round(river_level, 2),
        'Soil_Moisture_%': round(soil_moisture, 2),
        'Wind_Speed_kmh': round(wind_speed, 2),
        'Is_Monsoon': is_monsoon,
        'Flood_Risk': flood_risk,
        'Flood_Occurred': flood_occurred
    })

# Create DataFrame
df = pd.DataFrame(data)

print(f"✓ Dataset created with {len(df)} samples")
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head(10)

In [ ]:
# Save the dataset
df.to_csv('flood_data_bihar_regions.csv', index=False)
print("✓ Dataset saved as 'flood_data_bihar_regions.csv'")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("Dataset Information:")
print("=" * 50)
df.info()
print("\n" + "=" * 50)
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check class distribution
print("\n" + "=" * 50)
print("Flood Risk Distribution:")
print(df['Flood_Risk'].value_counts())
print("\nFlood Occurrence Distribution:")
print(df['Flood_Occurred'].value_counts())

print("\n" + "=" * 50)
print("Region-wise Flood Occurrence:")
print(df.groupby('Region')['Flood_Occurred'].sum().sort_values(ascending=False))

In [ ]:
# Visualization 1: Flood Risk Distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Flood Risk distribution
df['Flood_Risk'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['green', 'orange', 'red'])
axes[0, 0].set_title('Flood Risk Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Risk Level')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=0)

# Flood occurrence by region
region_floods = df.groupby('Region')['Flood_Occurred'].sum().sort_values(ascending=True)
region_floods.plot(kind='barh', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Total Flood Occurrences by Region', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Number of Floods')
axes[0, 1].set_ylabel('Region')

# Monsoon vs Non-Monsoon floods
monsoon_floods = df.groupby('Is_Monsoon')['Flood_Occurred'].sum()
monsoon_floods.plot(kind='bar', ax=axes[1, 0], color=['skyblue', 'darkblue'])
axes[1, 0].set_title('Floods: Monsoon vs Non-Monsoon', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Monsoon (0=No, 1=Yes)')
axes[1, 0].set_ylabel('Flood Count')
axes[1, 0].tick_params(axis='x', rotation=0)

# Rainfall distribution
axes[1, 1].hist(df['Rainfall_7Day_mm'], bins=50, color='teal', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('7-Day Rainfall Distribution', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Rainfall (mm)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].axvline(df['Rainfall_7Day_mm'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('flood_distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Distribution analysis plots created")

In [ ]:
# Visualization 2: Feature Correlations
plt.figure(figsize=(14, 10))

# Select numerical features
numerical_features = ['Daily_Rainfall_mm', 'Rainfall_7Day_mm', 'Rainfall_15Day_mm', 
                     'Humidity_%', 'Temperature_C', 'River_Level_m', 
                     'Soil_Moisture_%', 'Wind_Speed_kmh', 'Flood_Occurred']

correlation_matrix = df[numerical_features].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Correlation heatmap created")

In [ ]:
# Visualization 3: Key Features vs Flood Occurrence
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

features_to_plot = [
    ('Rainfall_7Day_mm', '7-Day Rainfall (mm)'),
    ('River_Level_m', 'River Level (m)'),
    ('Humidity_%', 'Humidity (%)'),
    ('Soil_Moisture_%', 'Soil Moisture (%)'),
    ('Temperature_C', 'Temperature (°C)'),
    ('Wind_Speed_kmh', 'Wind Speed (km/h)')
]

for idx, (feature, title) in enumerate(features_to_plot):
    row = idx // 3
    col = idx % 3
    
    df.boxplot(column=feature, by='Flood_Occurred', ax=axes[row, col])
    axes[row, col].set_title(f'{title} vs Flood Occurrence', fontsize=12, fontweight='bold')
    axes[row, col].set_xlabel('Flood Occurred (0=No, 1=Yes)')
    axes[row, col].set_ylabel(title)
    plt.sca(axes[row, col])
    plt.xticks([1, 2], ['No', 'Yes'])

plt.suptitle('')  # Remove the default title
plt.tight_layout()
plt.savefig('features_vs_flood.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Feature analysis plots created")

## 4. Data Preprocessing

In [ ]:
# Prepare features (X) and target (y)
# We'll predict Flood_Occurred (binary classification)

# Encode Region as numerical
label_encoder = LabelEncoder()
df['Region_Encoded'] = label_encoder.fit_transform(df['Region'])

# Select features
feature_columns = [
    'Region_Encoded',
    'Daily_Rainfall_mm',
    'Rainfall_7Day_mm',
    'Rainfall_15Day_mm',
    'Humidity_%',
    'Temperature_C',
    'River_Level_m',
    'Soil_Moisture_%',
    'Wind_Speed_kmh',
    'Is_Monsoon'
]

X = df[feature_columns]
y = df['Flood_Occurred']

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nFeature names:")
print(feature_columns)

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nTraining set flood occurrence:")
print(y_train.value_counts())
print(f"\nTesting set flood occurrence:")
print(y_test.value_counts())

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled using StandardScaler")
print(f"\nScaled training data shape: {X_train_scaled.shape}")
print(f"Scaled testing data shape: {X_test_scaled.shape}")

## 5. Model Training and Evaluation

We'll train and compare multiple models:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting

In [ ]:
# Dictionary to store models and their performance
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5)
}

results = {}

print("Training models...\n")
print("=" * 70)

for name, model in models.items():
    print(f"\n{name}")
    print("-" * 70)
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    # Print results
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"CV Score:  {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print("\n" + "=" * 70)
print("✓ All models trained successfully!")

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'Precision': [results[m]['precision'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'F1-Score': [results[m]['f1'] for m in results.keys()],
    'CV Mean': [results[m]['cv_mean'] for m in results.keys()],
    'CV Std': [results[m]['cv_std'] for m in results.keys()]
})

print("\nModel Comparison:")
print("=" * 100)
print(comparison_df.to_string(index=False))
print("=" * 100)

# Find best model
best_model_name = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   F1-Score: {comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'F1-Score']:.4f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(comparison_df))
width = 0.2

for i, metric in enumerate(metrics):
    axes[0].bar(x + i*width, comparison_df[metric], width, label=metric)

axes[0].set_xlabel('Model', fontweight='bold')
axes[0].set_ylabel('Score', fontweight='bold')
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Cross-validation scores
axes[1].bar(comparison_df['Model'], comparison_df['CV Mean'], 
           yerr=comparison_df['CV Std'], capsize=5, color='skyblue', edgecolor='navy')
axes[1].set_xlabel('Model', fontweight='bold')
axes[1].set_ylabel('CV Score', fontweight='bold')
axes[1].set_title('Cross-Validation Scores', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Model comparison visualization created")

## 6. Detailed Analysis of Best Model

In [ ]:
# Select best model for detailed analysis
best_model = results[best_model_name]['model']
y_pred_best = results[best_model_name]['y_pred']

print(f"Detailed Analysis: {best_model_name}")
print("=" * 70)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, 
                          target_names=['No Flood', 'Flood']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(cm)

In [ ]:
# Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Flood', 'Flood'],
            yticklabels=['No Flood', 'Flood'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontweight='bold')
plt.xlabel('Predicted', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix visualization created")

In [ ]:
# Feature Importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nFeature Importance:")
    print(feature_importance.to_string(index=False))
    
    # Visualize feature importance
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='teal')
    plt.xlabel('Importance', fontweight='bold')
    plt.ylabel('Feature', fontweight='bold')
    plt.title(f'Feature Importance - {best_model_name}', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Feature importance visualization created")
else:
    print("\nFeature importance not available for this model type.")

In [ ]:
# ROC Curve
if results[best_model_name]['y_pred_proba'] is not None:
    fpr, tpr, thresholds = roc_curve(y_test, results[best_model_name]['y_pred_proba'])
    roc_auc = roc_auc_score(y_test, results[best_model_name]['y_pred_proba'])
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontweight='bold')
    plt.ylabel('True Positive Rate', fontweight='bold')
    plt.title(f'ROC Curve - {best_model_name}', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('roc_curve.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ ROC curve created (AUC = {roc_auc:.4f})")

## 7. Model Optimization (Hyperparameter Tuning)

In [ ]:
# Hyperparameter tuning for the best model
print(f"Optimizing {best_model_name}...\n")

if best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    base_model = RandomForestClassifier(random_state=42)
    
elif best_model_name == 'Gradient Boosting':
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 0.9, 1.0]
    }
    base_model = GradientBoostingClassifier(random_state=42)
    
elif best_model_name == 'Decision Tree':
    param_grid = {
        'max_depth': [5, 10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    }
    base_model = DecisionTreeClassifier(random_state=42)
    
else:  # Logistic Regression
    param_grid = {
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
    base_model = LogisticRegression(random_state=42, max_iter=1000)

# Grid Search
grid_search = GridSearchCV(
    base_model, param_grid, cv=5, scoring='f1', 
    n_jobs=-1, verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("\n" + "=" * 70)
print("Optimization Results:")
print("=" * 70)
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Get optimized model
optimized_model = grid_search.best_estimator_

In [ ]:
# Evaluate optimized model
y_pred_optimized = optimized_model.predict(X_test_scaled)

print("\nOptimized Model Performance:")
print("=" * 70)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_optimized):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_optimized):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_optimized):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_optimized):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_optimized, 
                          target_names=['No Flood', 'Flood']))

## 8. Save the Final Model

In [ ]:
# Save the optimized model, scaler, and label encoder
joblib.dump(optimized_model, 'flood_prediction_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')

# Save feature names
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

print("✓ Model saved as 'flood_prediction_model.pkl'")
print("✓ Scaler saved as 'scaler.pkl'")
print("✓ Label encoder saved as 'label_encoder.pkl'")
print("✓ Feature names saved as 'feature_names.pkl'")

# Save model metadata
metadata = {
    'model_name': best_model_name,
    'best_params': grid_search.best_params_,
    'accuracy': accuracy_score(y_test, y_pred_optimized),
    'precision': precision_score(y_test, y_pred_optimized),
    'recall': recall_score(y_test, y_pred_optimized),
    'f1_score': f1_score(y_test, y_pred_optimized),
    'features': feature_columns,
    'regions': list(regions.keys())
}

with open('model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("✓ Model metadata saved as 'model_metadata.pkl'")

## 9. Testing the Model with Sample Predictions

In [ ]:
# Create test scenarios
print("Testing Model with Sample Scenarios")
print("=" * 70)

# Scenario 1: High flood risk (Monsoon, heavy rainfall)
scenario_1 = pd.DataFrame([{
    'Region_Encoded': label_encoder.transform(['Darbhanga'])[0],
    'Daily_Rainfall_mm': 180,
    'Rainfall_7Day_mm': 650,
    'Rainfall_15Day_mm': 1200,
    'Humidity_%': 92,
    'Temperature_C': 29,
    'River_Level_m': 3.5,
    'Soil_Moisture_%': 95,
    'Wind_Speed_kmh': 25,
    'Is_Monsoon': 1
}])

# Scenario 2: Medium flood risk
scenario_2 = pd.DataFrame([{
    'Region_Encoded': label_encoder.transform(['Patna'])[0],
    'Daily_Rainfall_mm': 85,
    'Rainfall_7Day_mm': 320,
    'Rainfall_15Day_mm': 550,
    'Humidity_%': 78,
    'Temperature_C': 31,
    'River_Level_m': 0.8,
    'Soil_Moisture_%': 68,
    'Wind_Speed_kmh': 18,
    'Is_Monsoon': 1
}])

# Scenario 3: Low flood risk (Non-monsoon, low rainfall)
scenario_3 = pd.DataFrame([{
    'Region_Encoded': label_encoder.transform(['Gorakhpur'])[0],
    'Daily_Rainfall_mm': 5,
    'Rainfall_7Day_mm': 25,
    'Rainfall_15Day_mm': 40,
    'Humidity_%': 55,
    'Temperature_C': 20,
    'River_Level_m': -1.5,
    'Soil_Moisture_%': 35,
    'Wind_Speed_kmh': 12,
    'Is_Monsoon': 0
}])

scenarios = {
    'High Risk (Darbhanga, Heavy Monsoon)': scenario_1,
    'Medium Risk (Patna, Moderate Monsoon)': scenario_2,
    'Low Risk (Gorakhpur, Non-Monsoon)': scenario_3
}

for scenario_name, scenario_data in scenarios.items():
    # Scale the features
    scenario_scaled = scaler.transform(scenario_data)
    
    # Make prediction
    prediction = optimized_model.predict(scenario_scaled)[0]
    prediction_proba = optimized_model.predict_proba(scenario_scaled)[0]
    
    print(f"\n{scenario_name}:")
    print("-" * 70)
    print(f"  Prediction: {'FLOOD RISK' if prediction == 1 else 'NO FLOOD'}")
    print(f"  Confidence: No Flood = {prediction_proba[0]*100:.1f}%, Flood = {prediction_proba[1]*100:.1f}%")
    print(f"  Input Parameters:")
    for col in scenario_data.columns:
        if col != 'Region_Encoded':
            print(f"    {col}: {scenario_data[col].values[0]}")

print("\n" + "=" * 70)
print("✓ Model testing completed successfully!")

## 10. Function for Making Predictions (For Web Integration)

In [ ]:
def predict_flood_risk(region, daily_rainfall, rainfall_7day, rainfall_15day, 
                      humidity, temperature, river_level, soil_moisture, 
                      wind_speed, is_monsoon):
    """
    Predict flood risk based on input parameters.
    
    Parameters:
    -----------
    region : str
        Region name (e.g., 'Patna', 'Darbhanga')
    daily_rainfall : float
        Daily rainfall in mm
    rainfall_7day : float
        7-day cumulative rainfall in mm
    rainfall_15day : float
        15-day cumulative rainfall in mm
    humidity : float
        Humidity percentage
    temperature : float
        Temperature in Celsius
    river_level : float
        River level in meters (above danger mark)
    soil_moisture : float
        Soil moisture percentage
    wind_speed : float
        Wind speed in km/h
    is_monsoon : int
        1 if monsoon season, 0 otherwise
    
    Returns:
    --------
    dict : Prediction results with flood risk and confidence
    """
    
    # Load saved models
    model = joblib.load('flood_prediction_model.pkl')
    scaler = joblib.load('scaler.pkl')
    le = joblib.load('label_encoder.pkl')
    
    # Encode region
    try:
        region_encoded = le.transform([region])[0]
    except:
        return {"error": f"Unknown region: {region}. Available regions: {list(le.classes_)}"}
    
    # Create input dataframe
    input_data = pd.DataFrame([{
        'Region_Encoded': region_encoded,
        'Daily_Rainfall_mm': daily_rainfall,
        'Rainfall_7Day_mm': rainfall_7day,
        'Rainfall_15Day_mm': rainfall_15day,
        'Humidity_%': humidity,
        'Temperature_C': temperature,
        'River_Level_m': river_level,
        'Soil_Moisture_%': soil_moisture,
        'Wind_Speed_kmh': wind_speed,
        'Is_Monsoon': is_monsoon
    }])
    
    # Scale features
    input_scaled = scaler.transform(input_data)
    
    # Make prediction
    prediction = model.predict(input_scaled)[0]
    prediction_proba = model.predict_proba(input_scaled)[0]
    
    # Determine risk level
    flood_probability = prediction_proba[1] * 100
    
    if flood_probability >= 70:
        risk_level = "HIGH"
        color = "red"
    elif flood_probability >= 40:
        risk_level = "MEDIUM"
        color = "orange"
    else:
        risk_level = "LOW"
        color = "green"
    
    return {
        "region": region,
        "flood_predicted": bool(prediction),
        "flood_probability": round(flood_probability, 2),
        "no_flood_probability": round(prediction_proba[0] * 100, 2),
        "risk_level": risk_level,
        "risk_color": color,
        "message": f"{'Flood risk detected' if prediction else 'No immediate flood risk'} in {region}"
    }

# Test the function
print("Testing prediction function:\n")
test_result = predict_flood_risk(
    region='Darbhanga',
    daily_rainfall=150,
    rainfall_7day=600,
    rainfall_15day=1100,
    humidity=90,
    temperature=28,
    river_level=2.5,
    soil_moisture=88,
    wind_speed=22,
    is_monsoon=1
)

print("Prediction Result:")
for key, value in test_result.items():
    print(f"  {key}: {value}")

print("\n✓ Prediction function ready for web integration!")

## 11. Model Summary and Next Steps

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════╗
║          FLOOD PREDICTION MODEL - PROJECT SUMMARY                     ║
╚═══════════════════════════════════════════════════════════════════════╝

📊 DATASET:
   • Total samples: 5,000
   • Regions covered: 10 (Bihar, Assam, UP, West Bengal)
   • Features: 10 (weather + geographical parameters)
   • Target: Flood occurrence (binary classification)

🤖 MODEL:
   • Best algorithm: """ + best_model_name + """
   • Accuracy: """ + f"{accuracy_score(y_test, y_pred_optimized):.2%}" + """
   • F1-Score: """ + f"{f1_score(y_test, y_pred_optimized):.4f}" + """
   • Status: Trained, optimized, and saved ✓

📁 OUTPUT FILES:
   ✓ flood_prediction_model.pkl (trained model)
   ✓ scaler.pkl (feature scaler)
   ✓ label_encoder.pkl (region encoder)
   ✓ model_metadata.pkl (model information)
   ✓ flood_data_bihar_regions.csv (dataset)
   ✓ Visualization images (.png files)

🌐 WEB INTEGRATION:
   • Use the predict_flood_risk() function
   • Input: Weather parameters + region
   • Output: Flood risk level + probability
   • Ready for API deployment ✓

🎯 NEXT STEPS FOR WEB DASHBOARD:
   1. Create Flask/Django backend
   2. Load model using joblib
   3. Create API endpoint with predict_flood_risk()
   4. Build frontend with HTML/CSS/JavaScript
   5. Add interactive map for region selection
   6. Display predictions with color-coded risk levels
   7. Optional: Add historical data visualization

💡 SAMPLE BACKEND CODE (Flask):
   ```python
   from flask import Flask, request, jsonify
   import joblib
   
   app = Flask(__name__)
   model = joblib.load('flood_prediction_model.pkl')
   
   @app.route('/predict', methods=['POST'])
   def predict():
       data = request.json
       result = predict_flood_risk(**data)
       return jsonify(result)
   ```

════════════════════════════════════════════════════════════════════════
                    ✓ MODEL DEVELOPMENT COMPLETE
════════════════════════════════════════════════════════════════════════
""")

print("\n🎉 Project successfully completed!")
print("📧 Ready for web dashboard development.")
print("\nAll files have been saved in the current directory.")